#### Transform Refunds Data

1. Extract specific portion of string from refund_reason using split function
2. Extract specific portion of string from refund_reason using regexp_extract function
3. Extract Date and Time from refund_timestamp
4. Write transformed data to the Silver schema

In [0]:
df_refunds = spark.table('gizmobox_catalog_subbu.bronze.py_refunds')
display(df_refunds)

##### 1. Extract specific portion of string from refund_reason using split function

In [0]:
from pyspark.sql import functions as F
df_split_funds = (
    df_refunds
    .select(
        'refund_id',
        'payment_id',
        'refund_timestamp',
        'refund_amount',
         F.split('refund_reason',':')[0].alias('refund_reason'),
         F.split('refund_reason',':')[1].alias('refund_source')
    ) 
)
display(df_split_funds)

##### 2. Extract specific portion of string from refund_reason using regexp_extract function

In [0]:
from pyspark.sql import functions as F
df_transformed_refunds = (
    df_refunds
    .select(
        'refund_id',
        'payment_id',
        F.date_format('refund_timestamp','yyyy-MM-dd').cast('date').alias('refund_timestamp'),
        F.date_format('refund_timestamp','HH:mm:ss').alias('refund_time'),
        'refund_amount',
         F.regexp_extract('refund_reason','^([^:]+):',1).alias('refund_reason'),
         F.regexp_extract('refund_reason','^[^:]+:(.*)$',1).alias('refund_source')
    ) 
)
display(df_transformed_refunds)

##### 3. Write transformed data to the Silver schema

In [0]:
df_transformed_refunds.writeTo('gizmobox_catalog_subbu.silver.py_refunds').createOrReplace()


In [0]:
%sql
select * from gizmobox_catalog_subbu.silver.py_refunds